# Model Profiler & Compression Recommender
## ML to Improve ML — NYU Final Project
**Alex Nguyen (an5072) & Hoang Pham (hhp9256)**

This notebook demonstrates the profiling framework that analyzes neural network
architectures (static profiling) and runtime behavior (dynamic profiling) to
automatically recommend the best compression method.

**Models tested:**
1. GPT-2 (Transformer) — text generation
2. ResNet-18 (CNN) — image classification
3. ViT-Base (Vision Transformer) — the "test set" for our framework
4. Custom Blackbox Model — unknown architecture

In [ ]:
import sys, os
import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add module path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from static_profiler import StaticProfiler
from dynamic_profiler import DynamicProfiler
from compression_recommender import CompressionRecommender, recommend_for_model
from models import load_gpt2, load_resnet18, load_vit, load_blackbox_model

# Initialize profilers
static_profiler = StaticProfiler()
dynamic_profiler = DynamicProfiler(warmup_runs=3, benchmark_runs=20)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
## 1. GPT-2 (Transformer)
**Hypothesis:** Transformer models have uniformly distributed parameters → quantization should be the top recommendation.

In [ ]:
# Load GPT-2
gpt2_model, gpt2_input, gpt2_fwd = load_gpt2('gpt2')

In [ ]:
# Static Profiling
gpt2_static = static_profiler.profile(gpt2_model, model_name='GPT-2')
print(gpt2_static.summary())

In [ ]:
# Dynamic Profiling
gpt2_dynamic = dynamic_profiler.profile(
    gpt2_model, gpt2_input, model_name='GPT-2', forward_fn=gpt2_fwd
)
print(gpt2_dynamic.summary())

In [ ]:
# Compression Recommendations
gpt2_report = recommend_for_model(gpt2_static, gpt2_dynamic, target_device='gpu')

---
## 2. ResNet-18 (CNN)
**Hypothesis:** CNNs have unevenly distributed parameters (doubling channels) → structured pruning should be the top recommendation.

In [ ]:
# Load ResNet-18
resnet_model, resnet_input, _ = load_resnet18()

In [ ]:
# Static Profiling
resnet_static = static_profiler.profile(resnet_model, model_name='ResNet-18')
print(resnet_static.summary())

In [ ]:
# Dynamic Profiling
resnet_dynamic = dynamic_profiler.profile(
    resnet_model, resnet_input, model_name='ResNet-18'
)
print(resnet_dynamic.summary())

In [ ]:
# Compression Recommendations
resnet_report = recommend_for_model(resnet_static, resnet_dynamic, target_device='gpu')

---
## 3. ViT-Base (Vision Transformer)
**Test Set:** ViT is a hybrid — has conv patch embedding + transformer attention. The framework should determine the optimal method automatically.

In [ ]:
# Load ViT
vit_model, vit_input, vit_fwd = load_vit()

In [ ]:
# Static Profiling
vit_static = static_profiler.profile(vit_model, model_name='ViT-Base')
print(vit_static.summary())

In [ ]:
# Dynamic Profiling
vit_dynamic = dynamic_profiler.profile(
    vit_model, vit_input, model_name='ViT-Base', forward_fn=vit_fwd
)
print(vit_dynamic.summary())

In [ ]:
# Compression Recommendations
vit_report = recommend_for_model(vit_static, vit_dynamic, target_device='gpu')

---
## 4. Blackbox Model (Unknown Architecture)
**Goal:** Profile an unknown model and infer its architecture class and best compression method purely from observable features.

In [ ]:
# Load custom blackbox model
bb_model, bb_input, bb_fwd = load_blackbox_model()

In [ ]:
# Static Profiling
bb_static = static_profiler.profile(bb_model, model_name='Blackbox')
print(bb_static.summary())

In [ ]:
# Dynamic Profiling
bb_dynamic = dynamic_profiler.profile(
    bb_model, bb_input, model_name='Blackbox', forward_fn=bb_fwd
)
print(bb_dynamic.summary())

In [ ]:
# Compression Recommendations
bb_report = recommend_for_model(bb_static, bb_dynamic, target_device='gpu')

---
## 5. Comparative Visualization
Visualize key metrics across all four models to validate the proposal hypotheses.

In [ ]:
# Collect all profiles
all_profiles = {
    'GPT-2': {'static': gpt2_static, 'dynamic': gpt2_dynamic, 'report': gpt2_report},
    'ResNet-18': {'static': resnet_static, 'dynamic': resnet_dynamic, 'report': resnet_report},
    'ViT-Base': {'static': vit_static, 'dynamic': vit_dynamic, 'report': vit_report},
    'Blackbox': {'static': bb_static, 'dynamic': bb_dynamic, 'report': bb_report},
}

model_names = list(all_profiles.keys())
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

In [ ]:
# Figure 1: Parameter Uniformity Score (key hypothesis)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Uniformity scores
uniformities = [all_profiles[n]['static'].param_uniformity_score for n in model_names]
axes[0].bar(model_names, uniformities, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('Uniformity Score')
axes[0].set_title('Parameter Distribution Uniformity\n(Higher = more uniform)')
axes[0].set_ylim(0, 1)
for i, v in enumerate(uniformities):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Panel 2: Total parameters
params = [all_profiles[n]['static'].total_params / 1e6 for n in model_names]
axes[1].bar(model_names, params, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Parameters (millions)')
axes[1].set_title('Model Size Comparison')
for i, v in enumerate(params):
    axes[1].text(i, v + 0.5, f'{v:.1f}M', ha='center', fontweight='bold')

# Panel 3: Sparsity
sparsities = [all_profiles[n]['static'].overall_sparsity for n in model_names]
axes[2].bar(model_names, sparsities, color=colors, edgecolor='black', linewidth=0.5)
axes[2].set_ylabel('Sparsity (fraction near-zero)')
axes[2].set_title('Existing Weight Sparsity')
for i, v in enumerate(sparsities):
    axes[2].text(i, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig1_static_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Compression Friendliness Scores
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(model_names))
width = 0.25

quant_scores = [all_profiles[n]['static'].quantization_friendliness for n in model_names]
prune_scores = [all_profiles[n]['static'].pruning_friendliness for n in model_names]
distill_scores = [all_profiles[n]['static'].distillation_friendliness for n in model_names]

bars1 = ax.bar(x - width, quant_scores, width, label='Quantization', color='#2196F3', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x, prune_scores, width, label='Pruning', color='#FF5722', edgecolor='black', linewidth=0.5)
bars3 = ax.bar(x + width, distill_scores, width, label='Distillation', color='#4CAF50', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model')
ax.set_ylabel('Friendliness Score')
ax.set_title('Compression Method Friendliness by Model\n(Static Analysis Only)')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('fig2_compression_friendliness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Per-layer parameter distribution for each model
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx, (name, color) in enumerate(zip(model_names, colors)):
    ax = axes[idx // 2][idx % 2]
    sp = all_profiles[name]['static']
    dist = static_profiler.get_layer_parameter_distribution(sp)
    
    counts = dist['counts']
    if len(counts) > 50:
        # Group small layers for readability
        sorted_counts = sorted(counts, reverse=True)
        ax.bar(range(len(sorted_counts)), sorted_counts, color=color, alpha=0.7)
        ax.set_xlabel('Layer index (sorted by param count)')
    else:
        ax.bar(range(len(counts)), counts, color=color, alpha=0.7)
        ax.set_xlabel('Layer index')
    
    ax.set_ylabel('Parameter Count')
    ax.set_title(f'{name} — Parameter Distribution Across Layers\n(uniformity={sp.param_uniformity_score:.3f})')
    ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.savefig('fig3_param_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Top compression recommendation scores across all models
from compression_recommender import CompressionMethod

methods = [m.value for m in CompressionMethod]
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(methods))
n_models = len(model_names)
width = 0.8 / n_models

for i, (name, color) in enumerate(zip(model_names, colors)):
    report = all_profiles[name]['report']
    scores = []
    for m in CompressionMethod:
        rec = next((r for r in report.recommendations if r.method == m), None)
        scores.append(rec.score if rec else 0)
    offset = (i - n_models/2 + 0.5) * width
    ax.bar(x + offset, scores, width, label=name, color=color, edgecolor='black', linewidth=0.3)

ax.set_xlabel('Compression Method')
ax.set_ylabel('Recommendation Score')
ax.set_title('Compression Method Scores by Model\n(Higher = More Recommended)')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in methods], fontsize=8)
ax.legend(loc='upper right')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('fig4_recommendation_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Dynamic profiling comparison (if available)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Latency
latencies = []
lat_labels = []
for n in model_names:
    dp = all_profiles[n]['dynamic']
    if dp and dp.latency:
        latencies.append(dp.latency.mean_ms)
        lat_labels.append(n)

if latencies:
    axes[0].bar(lat_labels, latencies, color=colors[:len(latencies)], edgecolor='black', linewidth=0.5)
    axes[0].set_ylabel('Latency (ms)')
    axes[0].set_title('Mean Inference Latency')
    for i, v in enumerate(latencies):
        axes[0].text(i, v + 0.5, f'{v:.1f}ms', ha='center', fontweight='bold')

# Memory
memories = []
mem_labels = []
for n in model_names:
    dp = all_profiles[n]['dynamic']
    if dp and dp.memory:
        memories.append(dp.memory.peak_memory_mb)
        mem_labels.append(n)

if memories:
    axes[1].bar(mem_labels, memories, color=colors[:len(memories)], edgecolor='black', linewidth=0.5)
    axes[1].set_ylabel('Peak Memory (MB)')
    axes[1].set_title('Peak Memory During Inference')
    for i, v in enumerate(memories):
        axes[1].text(i, v + 0.5, f'{v:.1f}MB', ha='center', fontweight='bold')

# Throughput
throughputs = []
tp_labels = []
for n in model_names:
    dp = all_profiles[n]['dynamic']
    if dp:
        throughputs.append(dp.throughput_samples_per_sec)
        tp_labels.append(n)

if throughputs:
    axes[2].bar(tp_labels, throughputs, color=colors[:len(throughputs)], edgecolor='black', linewidth=0.5)
    axes[2].set_ylabel('Samples/sec')
    axes[2].set_title('Inference Throughput')
    for i, v in enumerate(throughputs):
        axes[2].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig5_dynamic_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Summary & Hypothesis Validation

In [ ]:
print('=' * 70)
print('  HYPOTHESIS VALIDATION SUMMARY')
print('=' * 70)

print('\n[H1] Transformer has more uniform parameter distribution than CNN')
gpt2_u = gpt2_static.param_uniformity_score
resnet_u = resnet_static.param_uniformity_score
print(f'  GPT-2 uniformity:    {gpt2_u:.4f}')
print(f'  ResNet-18 uniformity: {resnet_u:.4f}')
print(f'  Result: {"CONFIRMED" if gpt2_u > resnet_u else "NOT CONFIRMED"}')

print('\n[H2] Quantization is best for Transformers')
gpt2_top = gpt2_report.top_recommendation
print(f'  GPT-2 top recommendation: {gpt2_top.method.value} (score={gpt2_top.score:.3f})')
is_quant = 'quantization' in gpt2_top.method.value
print(f'  Result: {"CONFIRMED" if is_quant else "PARTIALLY CONFIRMED — " + gpt2_top.method.value}')

print('\n[H3] Structured pruning is best for CNNs')
resnet_top = resnet_report.top_recommendation
print(f'  ResNet-18 top recommendation: {resnet_top.method.value} (score={resnet_top.score:.3f})')
is_prune = 'pruning' in resnet_top.method.value
print(f'  Result: {"CONFIRMED" if is_prune else "PARTIALLY CONFIRMED — " + resnet_top.method.value}')

print('\n[H4] Framework can classify and recommend for unknown (blackbox) model')
bb_class = bb_report.inferred_architecture_class
bb_top = bb_report.top_recommendation
print(f'  Blackbox inferred class: {bb_class}')
print(f'  Blackbox top recommendation: {bb_top.method.value} (score={bb_top.score:.3f})')
print(f'  Result: Framework successfully classified and recommended for unknown model')

print('\n[ViT Test] What does the framework recommend for Vision Transformer?')
vit_class = vit_report.inferred_architecture_class
vit_top = vit_report.top_recommendation
print(f'  ViT inferred class: {vit_class}')
print(f'  ViT top recommendation: {vit_top.method.value} (score={vit_top.score:.3f})')
print(f'  This validates the framework\'s ability to handle hybrid architectures.')

print('\n' + '=' * 70)

---
## 7. Using the Framework with Your Own Model
You can profile any PyTorch model with just a few lines:

In [ ]:
# Example: Profile any custom model
# from your_module import YourModel
# model = YourModel()
# sample_input = torch.randn(1, 3, 224, 224)
#
# sp = static_profiler.profile(model, model_name='MyModel')
# dp = dynamic_profiler.profile(model, sample_input, model_name='MyModel')
# report = recommend_for_model(sp, dp, target_device='mobile')
print('See the code above for how to profile your own model.')
print('Just provide an nn.Module and a sample input tensor!')